### 프로그램 흐름
1. 문장 입력 (임시로 input()처리 해뒀지만 추후 프론트에서 전달받은 문장으로 진행)
2. 토큰화 및 모델 돌리기 -> 결과물로 감정 단어 리스트 출력
4. 감정 단어 분석하여 감정 도출 -> 데이터베이스에 저장
5. 이후 모듈은 무조건 데이터베이스에 접근하여 구현

### 프로그램 구조 예시
project/
├── frontend.py  <- 프론트 (파일 더 있어도 상관없어요!)  
├── tokenizer_model.py  <- 토크나이저와 모델 & 감정 단어 분석해서 최종 감정 도출  
├── emotion_db.py  <- 데이터베이스 구현용 모듈  
├── graphs.py  <- 그래프 만들거나 그런 거 하는 모듈  
└── data/  
    └── emotions.json  <- 데이터베이스  
    └── CONSTANTS.json  <- 뭐라고 해야 할 지 몰라서 일단 상수라고 했는데 감정문장이나 그런 불변하는 데이터 저장하는 DB입니다  

In [ ]:
!pip install transformers
!pip install torch

In [ ]:
from transformers import ElectraTokenizer, ElectraForSequenceClassification
import torch

# 토크나이저와 모델 불러오기 (감정 분석용 fine-tuned 모델 사용 가능)
tokenizer = ElectraTokenizer.from_pretrained("monologg/koelectra-base-v3-discriminator")
model = ElectraForSequenceClassification.from_pretrained("monologg/koelectra-base-v3-discriminator")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/61.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/263k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/467 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/452M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/452M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraForSequenceClassification LOAD REPORT from: monologg/koelectra-base-v3-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
electra.embeddings.position_ids                   | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
classifier.dense.weight                           | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missin

In [ ]:
EMOTIONS_LIST = ("행복/만족", "우울/피곤", "불안/걱정", "분노/짜증", "설렘/흥분", "외로움/공허", "평온/안정") # 원래 constants.py를 임포트해야하는데 임시로 여기 넣어요

In [ ]:
sentence = input()
inputs = tokenizer(sentence, return_tensors="pt")

outputs = model(**inputs)
logits = outputs.logits
predicted_class = torch.argmax(logits).item()

print("예측된 클래스:", predicted_class)

정말 행복하고 기분이 좋아


NameError: name 'tokenizer' is not defined

In [ ]:
# 방법 1: 토큰 별 감정 점수 계산
tokens = tokenizer.tokenize(sentence)
inputs = tokenizer(sentence, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

# logits에서 가장 강한 감정 클래스 확인
logits = outputs.logits
predicted_class = torch.argmax(logits, dim=-1).item()

print("문장 전체 감정:", predicted_class)
print("토큰들:", tokens)

문장 전체 감정: 1
토큰들: ['오늘', '너무', '행복', '##하', '##고', '기분', '##이', '좋', '##아', '!']


In [ ]:
# 방법 2: 사전 기반 감정 단어 필터링
emotion_words = ["행복", "기분", "좋아", "슬프다", "화나다", "즐겁다"]
extracted = [tok for tok in tokens if tok in emotion_words]
print("감정 단어 추출:", extracted)

감정 단어 추출: ['행복', '기분']


데이터베이스

In [ ]:
# emotions_db.py

import json
from datetime import datetime
from pathlib import Path

DB_PATH = Path("data/emotions.json")

def init_db():
    if not DB_PATH.exists():
        with open(DB_PATH, "w", encoding="utf-8") as f:
            json.dump([], f)

def add_emotion(emotion: str):
    with open(DB_PATH, "r+", encoding="utf-8") as f:
        data = json.load(f)
        new_entry = {
            "id": len(data) + 1,
            "date": datetime.now().strftime("%Y-%m-%d"),
            "emotion": emotion
        }
        data.append(new_entry)
        f.seek(0)
        json.dump(data, f, indent=4, ensure_ascii=False)

def get_all_emotions():
    with open(DB_PATH, "r", encoding="utf-8") as f:
        return json.load(f)


데이터베이스 형태 예시 (구현하실 때 일단 이거 가져다가 쓰시면 될 것 같아요)

In [ ]:
# emotions.json

[
  {
    "id": 1,
    "date": "2026-06-06",
    "emotion": "행복/만족"
  },
  {
    "id": 2,
    "date": "2026-06-07",
    "emotion": "우울/피곤"
  },
  {
    "id": 3,
    "date": "2026-06-08",
    "emotion": "불안/걱정"
  },
  {
    "id": 4,
    "date": "2026-06-09",
    "emotion": "행복/만족"
  },
]


In [ ]:
# CONSTANTS.py

EMOTIONS_LIST = ("행복/만족", "우울/피곤", "불안/걱정", "분노/짜증", "설렘/흥분", "외로움/공허", "평온/안정")

RELATING_WORDS_LIST = {     #임시로 짠 거라 구조 이상하면 바꿀 수도 있어요
    "행복/만족":{"예시문장1", "예시문장2"},
    "우울/피곤":{},
    "불안/걱정":{},
    "분노/짜증":{},
    "설렘/흥분":{},
    "외로움/공허":{},
    "평온/안정":{}
}